# The PyTorch training loop

_PyTorch (a complete course)_

**The hand-written loop at the heart of every PyTorch project.**

Training a network is a simple cycle repeated many times: guess, measure the error, find which way to nudge each weight, nudge. The loop just runs that cycle over your data, again and again.

---

This notebook is a practice scaffold for the **The PyTorch training loop** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)                       # reproducible
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- tiny synthetic 2-class dataset (two Gaussian blobs in 8-D) ---
N, D = 1000, 8
centers = torch.randn(2, D) * 2.0
y = torch.randint(0, 2, (N,))              # class labels 0 / 1
X = centers[y] + torch.randn(N, D)         # features near each class center

# train / validation split
ntr = 800
train_ds = TensorDataset(X[:ntr], y[:ntr])
val_ds   = TensorDataset(X[ntr:], y[ntr:])
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=64)

# --- model, loss, optimizer ---
model = nn.Sequential(
    nn.Linear(D, 32), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(32, 2),                      # raw logits (no softmax!)
).to(device)
criterion = nn.CrossEntropyLoss()          # expects logits + int labels
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

def run_epoch(loader, train):
    model.train() if train else model.eval()   # dropout/batchnorm mode
    total_loss, correct, n = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)   # move to device
            if train:
                optimizer.zero_grad()               # clear old grads
            pred = model(xb)                        # forward
            loss = criterion(pred, yb)              # measure error
            if train:
                loss.backward()                     # backprop
                optimizer.step()                    # update weights
            total_loss += loss.item() * len(xb)     # .item() -> no graph
            correct += (pred.argmax(1) == yb).sum().item()
            n += len(xb)
    return total_loss / n, correct / n

EPOCHS = 15
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_dl, train=True)
    va_loss, va_acc = run_epoch(val_dl,   train=False)
    print(f"epoch {epoch:2d} | "
          f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f}")


## Visualize the data & results

_Across epochs, does training loss keep falling while validation loss eventually turns back up?_

In [ ]:
import numpy as np

rng = np.random.RandomState(1)
d = 40                                 # 40 features...
ntr, nva = 30, 300                     # ...but tiny train set -> easy to overfit
w_true = np.zeros(d); w_true[:3] = [1.6, -1.4, 1.0]   # only 3 real signals

def make(n):
    X = rng.randn(n, d)
    logits = X @ w_true + 0.6 * rng.randn(n)
    return X, (logits > 0).astype(float)

Xtr, ytr = make(ntr)
Xva, yva = make(nva)
Xtr = np.hstack([Xtr, np.ones((ntr, 1))])   # bias column
Xva = np.hstack([Xva, np.ones((nva, 1))])

w = np.zeros(d + 1)
sig = lambda z: 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))
def bce(X, y, w):
    p = np.clip(sig(X @ w), 1e-7, 1 - 1e-7)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

lr, epochs = 0.3, 120
train_loss, val_loss = [], []
for e in range(epochs):
    p = sig(Xtr @ w)
    grad = Xtr.T @ (p - ytr) / ntr      # logistic gradient
    w -= lr * grad                      # the "optimizer.step()"
    train_loss.append(bce(Xtr, ytr, w))
    val_loss.append(bce(Xva, yva, w))   # held-out: no weight update

# subsample to <= 60 points for the chart (every 2nd epoch)
es = range(0, epochs, 2)
print("train:", [(e + 1, round(train_loss[e], 4)) for e in es])
print("val  :", [(e + 1, round(val_loss[e], 4)) for e in es])
print("val min at epoch", int(np.argmin(val_loss)) + 1,
      "=", round(min(val_loss), 3))


## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Type this in Colab. Write the five-line training step in the correct order on a tiny model. With
            torch.manual_seed(0), build model = nn.Linear(3, 1),
            opt = torch.optim.SGD(model.parameters(), lr=0.1), crit = nn.MSELoss(),
            x = torch.randn(5, 3), y = torch.randn(5, 1). Run exactly one batch:
            zero_grad &rarr; forward &rarr; loss &rarr; backward &rarr; step. Print the loss.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Order it: opt.zero_grad(), pred = model(x), loss = crit(pred, y), loss.backward(), opt.step(). — _Forward builds the autograd graph; backward fills .grad; step uses them — the order is load-bearing._
- Print loss.item() after the forward. — _.item() pulls out the plain Python number with no graph attached._

**Answer:** import torch
import torch.nn as nn
torch.manual_seed(0)
model = nn.Linear(3, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
crit = nn.MSELoss()
x = torch.randn(5, 3)
y = torch.randn(5, 1)

opt.zero_grad()          # 1. clear old grads
pred = model(x)          # 2. forward
loss = crit(pred, y)     # 3. measure error
loss.backward()          # 4. backprop -> fills .grad
opt.step()               # 5. update weights
print(round(loss.item(), 4))   # 1.0855

</details>

**Problem 2.** Type this in Colab. Prove that forgetting zero_grad() corrupts training. Make
            w = torch.zeros(1, requires_grad=True) and opt = torch.optim.SGD([w], lr=0.0).
            Loop 3 times: forward loss = (w - 2.0).pow(2), loss.backward(), but NEVER call
            zero_grad(). Print w.grad each time and watch it pile up.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Skip opt.zero_grad() inside the loop. — _PyTorch ADDS each backward's gradient onto .grad, so it grows: -4, -8, -12._
- Use lr=0.0 so w never changes and the grad is the same each backward. — _It isolates the accumulation: only the missing reset changes the printed value._

**Answer:** w = torch.zeros(1, requires_grad=True)
opt = torch.optim.SGD([w], lr=0.0)
for i in range(3):
    loss = (w - 2.0).pow(2)   # d/dw = 2(w-2) = -4 at w=0
    loss.backward()           # NO zero_grad -> accumulates
    print(w.grad)             # tensor([-4.]) tensor([-8.]) tensor([-12.])

</details>

**Problem 3.** Type this in Colab. Now fix it: same setup as above, but call opt.zero_grad() at the top
            of each iteration. Print w.grad each time and confirm it stays at -4.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Add opt.zero_grad() before loss.backward(). — _Resetting the grad to zero each step means each backward sees only its own gradient._
- Compare with the previous task's growing values. — _The constant -4 shows the reset is what keeps every update correct._

**Answer:** w = torch.zeros(1, requires_grad=True)
opt = torch.optim.SGD([w], lr=0.0)
for i in range(3):
    opt.zero_grad()           # the fix
    loss = (w - 2.0).pow(2)
    loss.backward()
    print(w.grad)             # tensor([-4.]) tensor([-4.]) tensor([-4.])

</details>

**Problem 4.** Type this in Colab. Show that model.eval() changes dropout behavior. Build
            torch.manual_seed(0); m = nn.Dropout(0.5) and x = torch.ones(10). In
            m.train() mode, print m(x) twice (different each time, some zeros). Then call
            m.eval() and print m(x) (all ones — dropout off).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Call the module in .train() then .eval() mode. — _Train mode randomly zeroes half the units and scales the rest; eval mode passes the input through unchanged._
- Use a fixed input of ones so the effect is obvious. — _Zeros appear only in train mode, proving why you must switch to eval for validation._

**Answer:** torch.manual_seed(0)
m = nn.Dropout(0.5)
x = torch.ones(10)
m.train()
print(m(x))   # e.g. tensor([2.,0.,2.,2.,0.,...])  random zeros, survivors x2
print(m(x))   # different again
m.eval()
print(m(x))   # tensor([1.,1.,1.,1.,1.,1.,1.,1.,1.,1.])  dropout OFF

</details>

**Problem 5.** Type this in Colab. Show that torch.no_grad() stops graph building. Make
            w = torch.ones(1, requires_grad=True). Compute (w * 2).requires_grad normally,
            then compute it inside with torch.no_grad():. Predict each before running, then verify.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read .requires_grad on the output tensor in both contexts. — _Outside the block it is True (graph tracked); inside it is False (no graph)._
- Wrap inference / validation in torch.no_grad(). — _No graph means less memory and faster evaluation — the validation-pass habit._

**Answer:** w = torch.ones(1, requires_grad=True)
print((w * 2).requires_grad)        # True
with torch.no_grad():
    print((w * 2).requires_grad)    # False  -- no graph built

</details>

**Problem 6.** Type this in Colab. Move a model and a batch to the chosen device to avoid a mismatch. Set
            device = "cuda" if torch.cuda.is_available() else "cpu". Put model = nn.Linear(4, 2).to(device)
            and a batch x = torch.randn(16, 4). Run model(x) WITHOUT moving x
            (note the error on GPU), then move it with x.to(device) and run again.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Build the device string once and .to(device) both model and data. — _An op between a cuda tensor and a cpu tensor raises a device-mismatch error._
- Print out.shape after the corrected call. — _A (16, 2) output confirms both operands now share a device._

**Answer:** device = "cuda" if torch.cuda.is_available() else "cpu"
model = nn.Linear(4, 2).to(device)
x = torch.randn(16, 4)
# On GPU this raises:
#   RuntimeError: Expected all tensors to be on the same device
# out = model(x)
x = x.to(device)              # fix: move the batch too
out = model(x)
print(out.shape)              # torch.Size([16, 2])

</details>

**Problem 7.** Type this in Colab. Write a reusable run_epoch(loader, train) helper. With
            torch.manual_seed(0), make a TensorDataset from X = torch.randn(200, 8),
            y = torch.randint(0, 2, (200,)), a DataLoader(batch_size=32, shuffle=True), a
            model nn.Sequential(nn.Linear(8,16), nn.ReLU(), nn.Linear(16,2)),
            CrossEntropyLoss, and Adam. The helper sets train()/eval(), wraps
            eval in no_grad, accumulates loss.item(), and returns mean loss + accuracy.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Toggle model.train() / model.eval() and pick enable_grad() vs no_grad() by the train flag. — _Train mode updates weights and builds the graph; eval mode skips both for correct, cheap validation._
- Accumulate loss.item() * len(xb), not the raw loss tensor. — _.item() drops the graph, avoiding the slow cross-epoch memory leak._

**Answer:** from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(0)
X = torch.randn(200, 8); y = torch.randint(0, 2, (200,))
dl = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)
model = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 2))
crit = nn.CrossEntropyLoss(); opt = torch.optim.Adam(model.parameters(), lr=1e-2)

def run_epoch(loader, train):
    model.train() if train else model.eval()
    ctx = torch.enable_grad() if train else torch.no_grad()
    total, correct, n = 0.0, 0, 0
    with ctx:
        for xb, yb in loader:
            if train: opt.zero_grad()
            pred = model(xb)
            loss = crit(pred, yb)
            if train:
                loss.backward(); opt.step()
            total += loss.item() * len(xb)            # .item() -> no graph
            correct += (pred.argmax(1) == yb).sum().item()
            n += len(xb)
    return total / n, correct / n

print(run_epoch(dl, train=True))    # e.g. (0.70..., 0.55...)

</details>

**Problem 8.** Type this in Colab. Put it together: train the model from the previous task for 5 epochs, calling
            run_epoch(dl, train=True) each epoch and printing the epoch number, mean loss, and accuracy.
            Confirm the loss falls across epochs.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Loop for epoch in range(1, 6): and call the helper. — _An epoch is one full pass over the loader; repeating it drives learning._
- Print the returned loss and accuracy each epoch. — _Watching loss fall (and accuracy rise) confirms the loop is wired correctly._

**Answer:** for epoch in range(1, 6):
    loss, acc = run_epoch(dl, train=True)
    print(f"epoch {epoch}  loss {loss:.4f}  acc {acc:.3f}")
# epoch 1  loss 0.70..  acc 0.5..
# epoch 5  loss 0.55..  acc 0.7..   (loss falls, accuracy climbs)

</details>